# Siamese exploration
This notebook will explore the lfw dataset and preprocess the dataset for training  

LFW datasert provides matching pairs of images as well as mismatched pairs, to train triplet loss, we need to generate triplets.  
To do this, we need to get every row of the dataset like this:  

**Anchor | Positive | Negative**  

In [66]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [67]:
# read in the datasets
lfw_path = "../data/lfw_dataset/"
img_folder = Path("../data/lfw_dataset/lfw-deepfunneled/lfw-deepfunneled")

df_match_train = pd.read_csv("../data/lfw_dataset/matchpairsDevTrain.csv")
df_mismatch_train = pd.read_csv("../data/lfw_dataset/mismatchpairsDevTrain.csv")

df_match_test = pd.read_csv("../data/lfw_dataset/matchpairsDevTest.csv")
df_mismatch_test = pd.read_csv("../data/lfw_dataset/mismatchpairsDevTest.csv")

print(df_match_train.columns)
print(df_mismatch_train.columns)

Index(['name', 'imagenum1', 'imagenum2'], dtype='str')
Index(['name', 'imagenum1', 'name.1', 'imagenum2'], dtype='str')


In [68]:
def get_lfw_zeros(number):
    """
    simple helper function to return enough zeros to have 4 digits in the file, 
    works only on positive numbers, it works for the purposes of this project
    """
    count = 0
    while number//(10**count) != 0:
        count += 1
    return (4 - count) * '0' + str(number)

In [69]:
# df_match_train['test']=df_match_train['imagenum1'].apply(lambda x: get_lfw_zeros(x))
# print(df_match_train['test'].head())
# df_match_train.drop(columns=['test'])
# print()

In [70]:
# This function is currently not used

# def get_path(path, filename):
#     """
#     simple helper function to find the path in a root path
#     returns a list of existing paths of filename under path

#     ** not currently used
#     """
#     res = list(Path(path).rglob(filename))
#     if not res or len(res) >1:
#         raise FileNotFoundError("u did something bad bro, we cant find the file")
#     return res[0]

In [71]:
def combine_cols(mode, pairs):
    """
    A function that combiens the columns of the pairs to:
    path_to_imgs | path_to_img1
    
    Args:
        mode (str): pairs have name | imagenum1 | imagenum2, and mismatch pairs have  name1 | imagenum1 | name | imagenum2, 
                    specify if you want 'match' or 'mistmatch'
        pairs (pd.Dataframe): the dataset of pairs
    """
    if mode not in ('match','mismatch'):
        print('u spelt match or mismatch wrong stupid')
        return 0
    mode = mode.upper()
    res = pairs.copy()

    if mode == 'MATCH':
        res['imagenum1'] = res['imagenum1'].apply(lambda x: get_lfw_zeros(x))
        res['imagenum2'] = res['imagenum2'].apply(lambda x: get_lfw_zeros(x))

        res['path1'] = res['name'] + '_' + res['imagenum1'] + '.jpg'
        res['path2'] = res['name'] + '_' + res['imagenum2'] + '.jpg'

    if mode == "MISMATCH":
        res['path1'] = res['name']
        res['path2'] = res['name.1']

        res['path1'] = res['path1'] + '_' + res['imagenum1'].apply(lambda x: get_lfw_zeros(x)) + '.jpg'
        res['path2'] = res['path2'] + '_' + res['imagenum2'].apply(lambda x: get_lfw_zeros(x)) + '.jpg'
        
    return res[['path1','path2']]

In [72]:
# combine_cols(lfw_path, 'match', df_match_train)

In [73]:
# combine_cols('mismatch', df_mismatch_test)

In [74]:
def keep_good(root, df):
    """
    drops all rows where the image that is not found 

    Args:
        root (str): the path that you want to check for the files
        df (pd.dataFrame): the df that you want to just segregate
    
    returns:
        res (pd.dataFrame): the df is is edited
    """
    root = Path(root)
    # applies a lambda function that checks if a file is in a certain root path
    res = df.copy()
    res['name1'] = res['path1'].apply(lambda name: name[:-9])
    res['name2'] = res['path2'].apply(lambda name: name[:-9])
    
    # build the path combine root with name and then with the other columns
    res['check1'] = root / res['name1'] / res['path1']
    res['check2'] = root / res['name2'] / res['path2']

    # drop the row if the file that are not found in the root path
    helper = lambda a: Path(a).is_file()
    res['check1'] = res['check1'].apply(helper)
    res['check2'] = res['check2'].apply(helper)

    res = res[res['check1'] & res['check2']]
    
    return res[['path1','path2']]

In [75]:
test_df = pd.DataFrame(
    np.array([['asjhdkashdj','asdasdasdasd'], 
              ['AJ_Cook_0001.jpg','Marsha_Thomason_0001.jpg'], 
              ['sum bullshit','Wayne_Ferreira_0005.jpg']]), 
              columns=["path1", "path2"]
)
test = Path("../data/lfw_dataset/lfw-deepfunneled/lfw-deepfunneled")
keep_good(test,test_df)

,path1,path2
1,AJ_Cook_0001.jpg,Marsha_Thomason_0001.jpg


In [76]:
def create_triplets(match, mismatch):
    """
    A function that creates triplets from two dfs

    Args:
        match (pd.DataFrame): the df with the matching paris
        missmatch (pd.DataFrame): the df with the mismatches

    Return:
        res (list): a list of sets, each element being unique
    """
    # rename the dfs
    m = match.rename(columns={'path1': 'anchor', 'path2': 'path2'})
    mm = mismatch.rename(columns={'path1': 'path2', 'path2': 'path3'})

    # left join on path2
    res = m.merge(mm, how='left', on='path2')
    res = res.rename(columns={'path2': 'positive', 'path3': 'negative'})
    res = res.dropna()

    return res

In [77]:
df_match_train = pd.read_csv("../data/lfw_dataset/matchpairsDevTrain.csv")
df_mismatch_train = pd.read_csv("../data/lfw_dataset/mismatchpairsDevTrain.csv")

df_match_train = keep_good(img_folder, combine_cols('match', df_match_train))
df_mismatch_train = keep_good(img_folder, combine_cols('mismatch', df_mismatch_train))

triplets = create_triplets(df_match_train, df_mismatch_train)
triplets

,anchor,positive,negative
2,Aaron_Sorkin_0001.jpg,Aaron_Sorkin_0002.jpg,Frank_Solich_0005.jpg
3,Abdel_Nasser_Assidi_0001.jpg,Abdel_Nasser_Assidi_0002.jpg,Hilary_McKay_0001.jpg
8,Abel_Pacheco_0001.jpg,Abel_Pacheco_0004.jpg,Lucio_Stanca_0001.jpg
9,Abel_Pacheco_0003.jpg,Abel_Pacheco_0004.jpg,Lucio_Stanca_0001.jpg
14,Adolfo_Aguilar_Zinser_0001.jpg,Adolfo_Aguilar_Zinser_0003.jpg,Soon_Yi_0001.jpg
...,...,...,...
893,Richard_Branson_0001.jpg,Richard_Branson_0002.jpg,Taufik_Hidayat_0001.jpg
897,Richard_Krajicek_0002.jpg,Richard_Krajicek_0003.jpg,Zalmay_Khalilzad_0001.jpg
954,Sam_Torrance_0002.jpg,Sam_Torrance_0003.jpg,Tom_Christerson_0001.jpg
1009,Terje_Roed-Larsen_0001.jpg,Terje_Roed-Larsen_0002.jpg,Thomas_Rupprath_0003.jpg


# another approach: making the triplets ourselves
make a list of all the people that have more than one picture, and then choose randomly two of them and pair them with a random picture of someone who is not them, pop both from the list and into a "used" dict, with keys of pair and alone. If one of the arrays runs out, start to reuse values inside the used dicts  

the last approach had some issues and had a lot of optimizations to approach what I think would be done using this method

In [78]:
all_names = pd.read_csv("../data/lfw_dataset/lfw_allnames.csv")
# initiate two dfs, one with multiple
multiple = all_names[all_names['images'] > 1]
solo = all_names[all_names['images'] == 1]

In [79]:
# changed the helper function from the previous one to fit this new approach

def get_lfw_zeros2(number):
    """
    simple helper function to return enough zeros to have 4 digits in the file, 
    works only on positive numbers, it works for the purposes of this project
    """
    count = 0
    while number//(10**count) != 0:
        count += 1
    return (4 - count) * '0'

In [80]:
def get_lfw_file_names(arr):
    """
    helper function for create_triplets. changes ['Aaron_Peirsol', 4] to 
    ['Aaron_Peirsol_0004.jpg','Aaron_Peirsol_0003.jpg','Aaron_Peirsol_0002.jpg','Aaron_Peirsol_0001.jpg']

    args:
        arr (list): the person, ie. ['Aaron_Peirsol', 4]
    
    returns:
        res (list): the list with the multiple people
        ['Aaron_Peirsol_0004.jpg','Aaron_Peirsol_0003.jpg','Aaron_Peirsol_0002.jpg','Aaron_Peirsol_0001.jpg']
    """
    person, number = arr
    zeros = get_lfw_zeros2(number)
    res = []
    while number != 0:
        res.append(person + "_" + str(zeros) + str(number) + ".jpg")
        number -= 1
    return res

In [119]:
import random 
def create_triplets(multiple, solo):
    """
    build triplets by choosing people from a list of multiple people, set two images of the same people to be the anchor and the positive,
    and then choose a third one from another list of people that only have one picture. 

    args:
        multiple (pd.dataFrame): list of people that have multiple people
        solo (pd.dataFrame): list of people that have only one picture

    return:
        res: a list of triplets
    """
    # flatten both dfs into lists so its easier to work with
    multiple = multiple.to_numpy().tolist()
    solo = solo.to_numpy().tolist()
    
    # convert all of them into their respective file names
    # in the multiple and solo list, in the respective lists like ['Aaron_Peirsol', 4], change it to 
    # ['Aaron_Peirsol_0004.jpg','Aaron_Peirsol_0003.jpg','Aaron_Peirsol_0002.jpg','Aaron_Peirsol_0001.jpg']
    # changes the solo into a dict early
    multiple = [get_lfw_file_names(a) for a in multiple]
    solo = dict.fromkeys([get_lfw_file_names(b)[0] for b in solo],0)
    res = []

    # turn the nested list into a list of dicts like, with the count of times it was used as the value:
    # {'Aaron_Peirsol_0004.jpg':0, ....... 'Aaron_Peirsol_0001.jpg':0}
    # do this for the solo dict, as we want the counts of individual photos too
    for i in multiple:
        i = dict.fromkeys(i, 0)
        while len(i)>1 and len(solo)>0:
            # choose two random elements inside i
            anchor, positive = random.sample(list(i.keys()), 2)

            # choose one random element inside solo
            negative = random.choice(list(solo.keys()))

            # append into res
            res.append([anchor, positive, negative])

            # pop if any element reaches 4 counts
            for d, key in [(i, anchor), (i, positive), (solo, negative)]:
                d[key] += 1
                if d[key] == 4:
                    d.pop(key)
    return res

In [146]:
res = create_triplets(multiple=multiple, solo=solo)
res[:4]

[['Aaron_Peirsol_0001.jpg',
  'Aaron_Peirsol_0002.jpg',
  'James_Meredeth_0001.jpg'],
 ['Aaron_Peirsol_0003.jpg',
  'Aaron_Peirsol_0001.jpg',
  'Stefan_Koubek_0001.jpg'],
 ['Aaron_Peirsol_0001.jpg',
  'Aaron_Peirsol_0004.jpg',
  'Yekaterina_Guseva_0001.jpg'],
 ['Aaron_Peirsol_0003.jpg', 'Aaron_Peirsol_0001.jpg', 'John_Daly_Jr_0001.jpg']]

In [154]:
def triplets_to_df(arr, max=10):
    """
    changes the nested list type that create_triplets make and change them to df,
    limiting each person to a maximum of max rows each

    args:
        arr (list): the nested list from create_triplets
        max (int): maximum amount of rows that each person is limited to
    
    return:
        res (pd.DataFrame): the resulting df from the filtering
    """
    res = pd.DataFrame(arr, columns = ['Anchor', 'Positive', 'Negative'])

    # make a row that gets rid of the last _000X.jpg portion so that only the name is present
    res['name'] = res['Anchor'].apply(lambda x: x[0:-9])
    res = res.groupby('name').head(max).reset_index(drop=True)
    return res

In [155]:
triplets_to_df(res)

,Anchor,Positive,Negative,name
0,Aaron_Peirsol_0001.jpg,Aaron_Peirsol_0002.jpg,James_Meredeth_0001.jpg,Aaron_Peirsol
1,Aaron_Peirsol_0003.jpg,Aaron_Peirsol_0001.jpg,Stefan_Koubek_0001.jpg,Aaron_Peirsol
2,Aaron_Peirsol_0001.jpg,Aaron_Peirsol_0004.jpg,Yekaterina_Guseva_0001.jpg,Aaron_Peirsol
3,Aaron_Peirsol_0003.jpg,Aaron_Peirsol_0001.jpg,John_Daly_Jr_0001.jpg,Aaron_Peirsol
4,Aaron_Peirsol_0003.jpg,Aaron_Peirsol_0002.jpg,Stephen_Crampton_0001.jpg,Aaron_Peirsol
...,...,...,...,...
10648,Thomas_OBrien_0006.jpg,Thomas_OBrien_0002.jpg,Esad_Landzo_0001.jpg,Thomas_OBrien
10649,Thomas_Rupprath_0002.jpg,Thomas_Rupprath_0003.jpg,John_Rusnak_0001.jpg,Thomas_Rupprath
10650,Thomas_Rupprath_0001.jpg,Thomas_Rupprath_0002.jpg,Ratna_Sari_Dewi_Sukarno_0001.jpg,Thomas_Rupprath
10651,Thomas_Rupprath_0001.jpg,Thomas_Rupprath_0002.jpg,Christopher_Amolsch_0001.jpg,Thomas_Rupprath
